# Problem 2

In [1]:
import os
import math
import time
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEEDS = [42, 123, 999]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FILE_LOCATION = '/content/drive/MyDrive/MTech/DNN Assignment 2/problem2/'

def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

results_file = '/content/drive/MyDrive/MTech/DNN Assignment 2/problem2/results.csv'
if not os.path.exists(results_file):
    df_init = pd.DataFrame(columns=[
        'problem', 'model', 'seed', 'seq_len', 'test_acc',
        'macro_f1', 'test_mse', 'params', 'sec_per_epoch',
        'val_loss_nats', 'bits_per_char', 'notes'
    ])
    df_init.to_csv(results_file, index=False)

def log_result(problem, model, seed, seq_len=None, test_acc=None, macro_f1=None,
               test_mse=None, params=None, sec_per_epoch=None, val_loss_nats=None,
               bits_per_char=None, notes=None):
    df = pd.read_csv(results_file)
    new_row = {
        'problem': problem, 'model': model, 'seed': seed, 'seq_len': seq_len,
        'test_acc': test_acc, 'macro_f1': macro_f1, 'test_mse': test_mse,
        'params': params, 'sec_per_epoch': sec_per_epoch,
        'val_loss_nats': val_loss_nats, 'bits_per_char': bits_per_char, 'notes': notes
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    df.to_csv(results_file, index=False)

os.makedirs(FILE_LOCATION+'figures', exist_ok=True)


## TASK 1: SCALED DOT-PRODUCT ATTENTION FROM SCRATCH

In [2]:
def scaled_dot_product_attention(q, k, v, mask=None):
    """
    q, k, v shapes: (B, H, T, d_k)
    mask shape: (1, 1, T, T) or (B, 1, T, T) where 1 indicates masked out positions (or boolean True for mask)
    """
    d_k = q.size(-1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_k)  # (B, H, T, T)

    if mask is not None:
        scores = scores.masked_fill(mask == 1, float('-inf'))

    attn_weights = F.softmax(scores, dim=-1)  # (B, H, T, T)
    # Handle NaN in softmax if an entire row is masked out
    attn_weights = torch.nan_to_num(attn_weights, nan=0.0)

    output = torch.matmul(attn_weights, v)  # (B, H, T, d_k)
    return output, attn_weights

# --- Unit Tests for Scaled Dot-Product Attention ---
def test_scaled_dot_product_attention():
    seed_everything(42)
    B, H, T, d_k = 2, 4, 8, 16
    q = torch.randn(B, H, T, d_k)
    k = torch.randn(B, H, T, d_k)
    v = torch.randn(B, H, T, d_k)

    # 1. Row sum verification
    out, weights = scaled_dot_product_attention(q, k, v)
    row_sums = weights.sum(dim=-1)
    assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5), "Attention weights row sums fail to equal 1.0"

    # 2. PyTorch reference equivalence verification
    out_ref = F.scaled_dot_product_attention(q, k, v)
    assert torch.allclose(out, out_ref, atol=1e-5), "Custom scaled dot product attention mismatches torch implementation"

    # 3. Causal mask verification
    causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool().unsqueeze(0).unsqueeze(0)
    _, causal_weights = scaled_dot_product_attention(q, k, v, mask=causal_mask)
    for i in range(T):
        for j in range(i + 1, T):
            assert torch.all(causal_weights[:, :, i, j] == 0.0), f"Causal mask failed at position ({i}, {j})"

    # 4. Hand-checkable 3-token verification
    q_hand = torch.tensor([[[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]]])  # (1, 1, 3, 2)
    k_hand = torch.tensor([[[[1.0, 0.0], [0.0, 1.0], [1.0, 0.0]]]])  # (1, 1, 3, 2)
    v_hand = torch.tensor([[[[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]]]])  # (1, 1, 3, 2)

    # Scale factor sqrt(2) approx 1.41421356
    # Scores for Token 0 wrt [K0, K1, K2] -> [1.0, 0.0, 1.0] / sqrt(2) = [0.7071, 0, 0.7071]
    # Softmax([0.7071, 0, 0.7071]) = [0.4223, 0.1554, 0.4223]
    # Expected V0 = 0.4223*[1,2] + 0.1554*[3,4] + 0.4223*[5,6] = [2.9999, 3.9999]
    out_h, w_h = scaled_dot_product_attention(q_hand, k_hand, v_hand)
    expected_w0 = F.softmax(torch.tensor([1.0, 0.0, 1.0]) / math.sqrt(2), dim=0)
    assert torch.allclose(w_h[0, 0, 0], expected_w0, atol=1e-4), "Hand check verification for weights failed"

    print("[PASS] Task 2.1: Attention unit tests and hand-checkable example passed.")

test_scaled_dot_product_attention()


[PASS] Task 2.1: Attention unit tests and hand-checkable example passed.



## TASK 2: MULTI-HEAD ATTENTION & ENCODER BLOCK


In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, d_model = x.size()

        # Project and reshape: (B, T, d_model) -> (B, num_heads, T, d_k)
        q = self.W_Q(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        k = self.W_K(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        v = self.W_V(x).view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        attn_out, attn_weights = scaled_dot_product_attention(q, k, v, mask=mask)

        # Reshape back: (B, num_heads, T, d_k) -> (B, T, d_model)
        concat_out = attn_out.transpose(1, 2).contiguous().view(B, T, d_model)
        output = self.W_O(concat_out)
        return output, attn_weights

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: (B, T, d_model)
        return x + self.pe[:, :x.size(1)]

class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-LayerNorm Architecture
        norm_x = self.ln1(x)
        attn_out, attn_weights = self.mha(norm_x, mask=mask)
        x = x + self.dropout(attn_out)

        norm_x2 = self.ln2(x)
        ffn_out = self.ffn(norm_x2)
        x = x + ffn_out
        return x, attn_weights

# --- Verify Parameter Count Derivation ---
def verify_parameter_counts():
    d_model, num_heads, d_ff = 128, 4, 512
    block = TransformerEncoderBlock(d_model, num_heads, d_ff)

    # Theoretical weight derivation:
    # W_Q, W_K, W_V, W_O = 4 * (d_model * d_model)
    # FFN weights = (d_model * d_ff) + (d_ff * d_model) = 2 * d_model * d_ff
    # Total pure weights = 4*d^2 + 2*d*d_ff = 4*(128^2) + 2*128*512 = 65,536 + 131,072 = 196,608

    # Biases and LayerNorm parameters:
    # MHA biases = 4 * d_model = 512
    # FFN biases = d_ff + d_model = 512 + 128 = 640
    # LayerNorm 1 & 2 = 2 * (2 * d_model) = 512
    # Total parameters in PyTorch = 196,608 + 512 + 640 + 512 = 198,272

    actual_params = sum(p.numel() for p in block.parameters() if p.requires_grad)
    print(f"[INFO] Theoretical block weights: 196,608 | Total block parameters (inc. biases/LN): {actual_params}")
    assert actual_params == 198272, f"Parameter count mismatch! Expected 198272, got {actual_params}"
    print("[PASS] Task 2.2: Transformer block parameter verification passed.")

verify_parameter_counts()

[INFO] Theoretical block weights: 196,608 | Total block parameters (inc. biases/LN): 198272
[PASS] Task 2.2: Transformer block parameter verification passed.


## TASK 3A: ENCODER CLASSIFIER ON HAR DATA




In [4]:
def load_har_data(data_dir='har/UCI HAR Dataset'):
    if not os.path.exists(data_dir):
        url = 'https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip'
        urllib.request.urlretrieve(url, 'har.zip')
        with zipfile.ZipFile('har.zip', 'r') as zip_ref:
            zip_ref.extractall('har')
        if os.path.exists('har/UCI HAR Dataset.zip'):
            with zipfile.ZipFile('har/UCI HAR Dataset.zip', 'r') as zip_ref:
                zip_ref.extractall('har')

    signal_names = [
        'body_acc_x_', 'body_acc_y_', 'body_acc_z_',
        'body_gyro_x_', 'body_gyro_y_', 'body_gyro_z_',
        'total_acc_x_', 'total_acc_y_', 'total_acc_z_'
    ]
    def load_signals(split):
        signals = []
        for name in signal_names:
            path = os.path.join(data_dir, split, 'Inertial Signals', f'{name}{split}.txt')
            signals.append(np.loadtxt(path))
        return np.stack(signals, axis=-1)

    X_train = load_signals('train')
    y_train = np.loadtxt(os.path.join(data_dir, 'train', 'y_train.txt')).astype(int) - 1
    sub_train = np.loadtxt(os.path.join(data_dir, 'train', 'subject_train.txt')).astype(int)

    X_test = load_signals('test')
    y_test = np.loadtxt(os.path.join(data_dir, 'test', 'y_test.txt')).astype(int) - 1
    sub_test = np.loadtxt(os.path.join(data_dir, 'test', 'subject_test.txt')).astype(int)

    return X_train, y_train, sub_train, X_test, y_test, sub_test

def get_har_splits(seed=42):
    X_train_raw, y_train_raw, sub_train_raw, X_test_raw, y_test_raw, _ = load_har_data()
    unique_subs = np.unique(sub_train_raw)
    rng = np.random.default_rng(seed)
    val_subs = rng.choice(unique_subs, 4, replace=False)

    val_mask = np.isin(sub_train_raw, val_subs)
    train_mask = ~val_mask

    X_tr, y_tr = X_train_raw[train_mask], y_train_raw[train_mask]
    X_va, y_va = X_train_raw[val_mask], y_train_raw[val_mask]
    X_te, y_te = X_test_raw, y_test_raw

    mean = X_tr.mean(axis=(0, 1), keepdims=True)
    std = X_tr.std(axis=(0, 1), keepdims=True) + 1e-8

    X_tr = (X_tr - mean) / std
    X_va = (X_va - mean) / std
    X_te = (X_te - mean) / std

    return (X_tr, y_tr), (X_va, y_va), (X_te, y_te)

class HAREncoderClassifier(nn.Module):
    def __init__(self, input_dim=9, d_model=128, num_heads=4, d_ff=512, num_layers=2,
                 num_classes=6, pe_type='sinusoidal', max_len=129):
        super().__init__()
        self.pe_type = pe_type
        self.input_proj = nn.Linear(input_dim, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        if pe_type == 'sinusoidal':
            self.pe = SinusoidalPositionalEncoding(d_model, max_len=max_len)
        elif pe_type == 'learned':
            self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        else:
            self.pe = None

        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B, T, _ = x.size()
        x = self.input_proj(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)  # (B, T+1, d_model)

        if self.pe_type == 'sinusoidal':
            x = self.pe(x)
        elif self.pe_type == 'learned':
            x = x + self.pe[:, :T+1]

        for layer in self.layers:
            x, _ = layer(x)

        cls_out = x[:, 0, :]
        return self.fc(cls_out)

def train_task_a_har():
    from sklearn.metrics import f1_score
    print("\n--- Running Task A: Encoder Classifier on HAR Data ---")

    for pe in ['sinusoidal', 'none', 'learned']:
        accs, f1s = [], []
        params = 0
        t0 = time.time()
        for seed in SEEDS:
            seed_everything(seed)
            (X_tr, y_tr), _, (X_te, y_te) = get_har_splits(seed=seed)
            train_loader = DataLoader(torch.utils.data.TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long)), batch_size=64, shuffle=True)
            test_loader = DataLoader(torch.utils.data.TensorDataset(torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te, dtype=torch.long)), batch_size=64, shuffle=False)

            model = HAREncoderClassifier(pe_type=pe).to(DEVICE)
            params = sum(p.numel() for p in model.parameters() if p.requires_grad)
            optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
            criterion = nn.CrossEntropyLoss()

            for epoch in range(20):
                model.train()
                for bx, by in train_loader:
                    bx, by = bx.to(DEVICE), by.to(DEVICE)
                    optimizer.zero_grad()
                    loss = criterion(model(bx), by)
                    loss.backward()
                    optimizer.step()

            model.eval()
            preds, targets = [], []
            with torch.no_grad():
                for bx, by in test_loader:
                    bx = bx.to(DEVICE)
                    preds.append(model(bx).argmax(dim=1).cpu().numpy())
                    targets.append(by.numpy())
            preds = np.concatenate(preds)
            targets = np.concatenate(targets)

            acc = (preds == targets).mean() * 100
            f1 = f1_score(targets, preds, average='macro')
            accs.append(acc)
            f1s.append(f1)

            log_result("P2_TaskA", f"Transformer_{pe}", seed, seq_len=128, test_acc=acc, macro_f1=f1, params=params)

        sec_per_epoch = (time.time() - t0) / (20 * len(SEEDS))
        print(f"PE Mode: {pe:10s} | Test Acc: {np.mean(accs):.2f}% +/- {np.std(accs):.2f}% | Macro-F1: {np.mean(f1s):.4f}")

train_task_a_har()


--- Running Task A: Encoder Classifier on HAR Data ---


/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA col

PE Mode: sinusoidal | Test Acc: 88.51% +/- 0.32% | Macro-F1: 0.8835


/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA col

PE Mode: none       | Test Acc: 87.89% +/- 2.23% | Macro-F1: 0.8765


/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


PE Mode: learned    | Test Acc: 88.79% +/- 0.88% | Macro-F1: 0.8853


/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


## TASK 4B: CHARACTER-LEVEL CAUSAL LANGUAGE MODEL

In [5]:
def download_tiny_shakespeare():
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    return urllib.request.urlopen(url).read().decode('utf-8')

text_data = download_tiny_shakespeare()
chars = sorted(list(set(text_data)))
vocab_size = len(chars)
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for i, ch in enumerate(chars)}
data_encoded = np.array([char2idx[ch] for ch in text_data], dtype=np.int64)

n_train = int(0.9 * len(data_encoded))
train_data = data_encoded[:n_train]
val_data = data_encoded[n_train:]

class CharDataset(Dataset):
    def __init__(self, data, context_len=128):
        self.data = data
        self.context_len = context_len

    def __len__(self):
        return len(self.data) - self.context_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.context_len]
        y = self.data[idx + 1 : idx + self.context_len + 1]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

class CausalTransformerLM(nn.Module):
    def __init__(self, vocab_size=65, d_model=128, num_heads=4, d_ff=512, num_layers=4, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pe = SinusoidalPositionalEncoding(d_model, max_len=max_len)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x, use_causal_mask=True):
        B, T = x.size()
        x = self.tok_emb(x) * math.sqrt(self.d_model)
        x = self.pe(x)

        mask = None
        if use_causal_mask:
            mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool().unsqueeze(0).unsqueeze(0)

        attn_maps = []
        for layer in self.layers:
            x, weights = layer(x, mask=mask)
            attn_maps.append(weights)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits, attn_maps

def sample_text(model, start_str="FIRST ", length=500, temperature=1.0, top_k=None):
    model.eval()
    input_ids = torch.tensor([char2idx[ch] for ch in start_str], dtype=torch.long).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        for _ in range(length):
            ctx = input_ids[:, -128:]
            logits, _ = model(ctx)
            logits = logits[:, -1, :] / max(temperature, 1e-5)

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            next_char = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_char], dim=1)

    return "".join([idx2char[i.item()] for i in input_ids[0]])

def train_language_model(epochs=5, batch_size=64):
    print("\n--- Running Task B: Character-Level Causal LM ---")
    seed_everything(42)
    train_dataset = CharDataset(train_data, context_len=128)
    val_dataset = CharDataset(val_data, context_len=128)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = CausalTransformerLM().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    train_losses, val_losses = [], []
    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        steps = 0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(bx)
            loss = criterion(logits.view(-1, vocab_size), by.view(-1))
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            steps += 1
            if steps >= 400: # Fast epoch truncation for Colab T4 budget
                break
        avg_train_loss = total_loss / steps
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        val_steps = 0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                logits, _ = model(bx)
                val_loss += criterion(logits.view(-1, vocab_size), by.view(-1)).item()
                val_steps += 1
                if val_steps >= 100:
                    break
        avg_val_loss = val_loss / val_steps
        val_losses.append(avg_val_loss)

        bpc = avg_val_loss / math.log(2)
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} nats | BPC: {bpc:.4f}")

    sec_per_epoch = (time.time() - t0) / epochs
    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log_result("P2_TaskB", "CausalTransformerLM", 42, seq_len=128, val_loss_nats=val_losses[-1], bits_per_char=val_losses[-1]/math.log(2), params=params, sec_per_epoch=sec_per_epoch)

    # Plot learning curve
    plt.figure(figsize=(7, 4))
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (nats)')
    plt.title('Tiny Shakespeare Causal LM Learning Curve')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FILE_LOCATION+'figures/lm_learning_curve.png')
    plt.close()

    # Generate and save text samples
    with open(FILE_LOCATION+'generated_samples.txt', 'w') as f:
        for temp in [0.2, 0.8, 1.5]:
            sample = sample_text(model, start_str="KING RICHARD: ", length=500, temperature=temp, top_k=40)
            header = f"=== Temperature {temp} ===\n"
            f.write(header + sample + "\n\n")
            print(f"\n{header}{sample[:200]}...\n")

    return model

lm_model = train_language_model(epochs=5)


--- Running Task B: Character-Level Causal LM ---
Epoch 01/05 | Train Loss: 2.5137 | Val Loss: 2.3931 nats | BPC: 3.4526
Epoch 02/05 | Train Loss: 2.2908 | Val Loss: 2.1825 nats | BPC: 3.1486
Epoch 03/05 | Train Loss: 2.0466 | Val Loss: 1.9693 nats | BPC: 2.8411
Epoch 04/05 | Train Loss: 1.8918 | Val Loss: 1.8605 nats | BPC: 2.6841
Epoch 05/05 | Train Loss: 1.7933 | Val Loss: 1.7748 nats | BPC: 2.5604


/tmp/ipykernel_404/3561889899.py:43: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)



=== Temperature 0.2 ===
KING RICHARD: the the he so the word
The see see the man the man the be to for the me to the he
And man the he her he her her hear so he grace
The see conter to my hear see the the seent
The man the p...


=== Temperature 0.8 ===
KING RICHARD: with lord you have no one your that getlemer on the
see in the coming in staly more, I'll not they of clouse?

CORIOLANUS:
Welos, and the from them anot,
Sto bee the boy hill hone love t...


=== Temperature 1.5 ===
KING RICHARD: In balse?

JULIEO:
Veight:
GRo, my hus, like! Yourt, her; boysold o't.
ISW:
Bnow,, perting fro, An y:--hear hone?
Cou's he'shallll'st! though: wilt, who fiy
trotuncnene, thekfuics doke,'...



## TASK 5: ANALYSIS & ABLATIONS

In [6]:
def run_transformer_ablations(model):
    print("\n--- Running Task 2.5: Analysis and Ablation Studies ---")

    # (a) Attention Heatmaps
    model.eval()
    sample_seq = torch.tensor([char2idx[ch] for ch in "ROMEO: Shall I hear more, or shall I speak at this?"], dtype=torch.long).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        _, attn_maps = model(sample_seq)

    # Last layer attention map
    last_layer_attn = attn_maps[-1][0].cpu().numpy()  # shape: (H, T, T)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    tokens = [ch for ch in "ROMEO: Shall I hear more, or shall I speak at this?"]

    for h_idx in range(2):
        im = axes[h_idx].imshow(last_layer_attn[h_idx], cmap='viridis')
        axes[h_idx].set_title(f'Layer 4 - Head {h_idx+1}')
        axes[h_idx].set_xticks(range(len(tokens)))
        axes[h_idx].set_yticks(range(len(tokens)))
        axes[h_idx].set_xticklabels(tokens, rotation=90, fontsize=6)
        axes[h_idx].set_yticklabels(tokens, fontsize=6)
    plt.colorbar(im, ax=axes.ravel().tolist())
    plt.tight_layout()
    plt.savefig(FILE_LOCATION+'figures/attention_heatmaps.png')
    plt.close()
    print("[INFO] Saved attention heatmaps to problem2/figures/attention_heatmaps.png")

    # (b) Heads Ablation Study
    print("\n[Ablation] Testing Heads = 1, 4, 8 at fixed d_model=128...")
    train_dataset = CharDataset(train_data, context_len=128)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    head_counts = [1, 4, 8]
    loss_histories = {}

    for h in head_counts:
        seed_everything(42)
        m = CausalTransformerLM(num_heads=h).to(DEVICE)
        opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        crit = nn.CrossEntropyLoss()

        m.train()
        step_losses = []
        step_count = 0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad()
            l, _ = m(bx)
            loss = crit(l.view(-1, vocab_size), by.view(-1))
            loss.backward()
            opt.step()

            step_losses.append(loss.item())
            step_count += 1
            if step_count >= 150:
                break

        loss_histories[h] = step_losses
        print(f"Heads: {h} | Final Loss: {np.mean(step_losses[-10:]):.4f}")

    # Generate & Save Heads Ablation Figure
    plt.figure(figsize=(7, 4.5))
    for h in head_counts:
        smoothed = np.convolve(loss_histories[h], np.ones(10)/10, mode='valid')
        plt.plot(smoothed, label=f'Heads = {h}')
    plt.xlabel('Training Steps')
    plt.ylabel('Training Loss (nats)')
    plt.title('Language Model Convergence Across Multi-Head Settings (d=128)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(FILE_LOCATION+'figures/heads_ablation.png')
    plt.close()
    print("[INFO] Saved heads ablation plot to problem2/figures/heads_ablation.png")

    # (c) Causal Mask Removal Ablation (With Comparison Plot)
    print("\n[Ablation] Removing Causal Mask (Cheating Ablation)...")
    modes = {'With Causal Mask (Standard)': True, 'Without Causal Mask (Cheating)': False}
    mask_losses = {}

    for label, use_mask in modes.items():
        seed_everything(42)
        m_mask = CausalTransformerLM().to(DEVICE)
        opt = torch.optim.Adam(m_mask.parameters(), lr=1e-3)
        crit = nn.CrossEntropyLoss()

        m_mask.train()
        step_losses = []
        step_count = 0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad()
            logits_out, _ = m_mask(bx, use_causal_mask=use_mask)
            loss = crit(logits_out.view(-1, vocab_size), by.view(-1))
            loss.backward()
            opt.step()

            step_losses.append(loss.item())
            step_count += 1
            if step_count >= 150:
                break
        mask_losses[label] = step_losses

    # Generate & Save Causal Mask Comparison Figure
    plt.figure(figsize=(7, 4.5))
    for label, losses in mask_losses.items():
        smoothed = np.convolve(losses, np.ones(5)/5, mode='valid')
        plt.plot(smoothed, label=label)
    plt.xlabel('Training Steps')
    plt.ylabel('Training Loss (nats)')
    plt.title('Effect of Removing Causal Mask on Training Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(FILE_LOCATION+'figures/no_mask_comparison.png')
    plt.close()
    print("[INFO] Saved no-mask comparison plot to problem2/figures/no_mask_comparison.png")

    # (d) Time & Memory Scaling vs Context Length
    print("\n[Scaling] Measuring Execution Time & GPU Peak Memory vs Context Length...")
    ctx_lengths = [64, 128, 256, 512]
    exec_times, mem_usages = [], []

    for ctx_len in ctx_lengths:
        dummy_input = torch.randint(0, vocab_size, (16, ctx_len)).to(DEVICE)
        m_scale = CausalTransformerLM(max_len=1024).to(DEVICE)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        t_start = time.time()
        for _ in range(10):
            out, _ = m_scale(dummy_input)
            loss = out.sum()
            loss.backward()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        else:
            mem_mb = 0.0

        elapsed = (time.time() - t_start) / 10
        exec_times.append(elapsed)
        mem_usages.append(mem_mb)
        print(f"Context: {ctx_len:4d} | Time per step: {elapsed*1000:.2f} ms | Peak Memory: {mem_mb:.2f} MB")

    fig, ax1 = plt.subplots(figsize=(7, 4))
    ax1.plot(ctx_lengths, exec_times, 'b-o', label='Time (s)')
    ax1.set_xlabel('Context Length (T)')
    ax1.set_ylabel('Execution Time per Step (s)', color='b')

    ax2 = ax1.twinx()
    ax2.plot(ctx_lengths, mem_usages, 'r-s', label='Memory (MB)')
    ax2.set_ylabel('Peak GPU Memory (MB)', color='r')

    plt.title('Execution Complexity vs Context Length O(T^2)')
    plt.tight_layout()
    plt.savefig(FILE_LOCATION+'figures/scaling_context_length.png')
    plt.close()
    print("[INFO] Saved context scaling plot to problem2/figures/scaling_context_length.png")

In [7]:
run_transformer_ablations(lm_model)

print("\n==========================================")
print("Problem 2 Execution Completed Successfully!")
print("Outputs logged to results.csv and generated_samples.txt")
print("==========================================")


--- Running Task 2.5: Analysis and Ablation Studies ---


/tmp/ipykernel_404/599907225.py:23: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


[INFO] Saved attention heatmaps to problem2/figures/attention_heatmaps.png

[Ablation] Testing Heads = 1, 4, 8 at fixed d_model=128...
Heads: 1 | Final Loss: 2.4805
Heads: 4 | Final Loss: 2.4691
Heads: 8 | Final Loss: 2.4693
[INFO] Saved heads ablation plot to problem2/figures/heads_ablation.png

[Ablation] Removing Causal Mask (Cheating Ablation)...
[INFO] Saved no-mask comparison plot to problem2/figures/no_mask_comparison.png

[Scaling] Measuring Execution Time & GPU Peak Memory vs Context Length...
Context:   64 | Time per step: 13.88 ms | Peak Memory: 113.67 MB
Context:  128 | Time per step: 13.13 ms | Peak Memory: 145.50 MB
Context:  256 | Time per step: 23.00 ms | Peak Memory: 318.13 MB
Context:  512 | Time per step: 71.33 ms | Peak Memory: 934.31 MB
[INFO] Saved context scaling plot to problem2/figures/scaling_context_length.png

Problem 2 Execution Completed Successfully!
Outputs logged to results.csv and generated_samples.txt
